In [1]:
from viewer.utils import load_env
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cmap

colormap = cmap.Colormap('tab20').to_matplotlib()
ks_output = Path(load_env('.env')["EPHYS_DATA_PATH"]) / "kilosort"

In [3]:
# LOAD PROBE CONTACTS 
ch_pos = np.load(ks_output / 'channel_positions.npy')
ch_shanks = np.load(ks_output / 'channel_shanks.npy').astype(int)
contacts_df = pd.DataFrame(ch_pos, columns=['x', 'y'])
contacts_df['shank'] = ch_shanks.astype(int)

# LOAD UNITS FROM KILOSORT OUTPUT
unit_labels = pd.read_csv(ks_output / 'cluster_group.tsv', sep='\t', index_col=0)
good_units = unit_labels[unit_labels['group'] == 'good'].index.values

clu = np.load(ks_output / 'spike_clusters.npy', mmap_mode='r')
spike_locations = np.load(ks_output / 'spike_positions.npy', mmap_mode='r')

good_mask = np.isin(clu, good_units)
spikes_df = pd.DataFrame(spike_locations[good_mask], columns=['x', 'y'])
spikes_df['cluster'] = clu[good_mask]
units_df = spikes_df.groupby('cluster').agg(['mean'])
units_df.columns = ['_'.join(col) for col in units_df.columns]

In [9]:
unit_xy = units_df[["x_mean", "y_mean"]].to_numpy(float)
contact_xy = contacts_df[["x", "y"]].to_numpy(float)

nearest_contact = np.argmin(
    ((unit_xy[:, None, :] - contact_xy[None, :, :]) ** 2).sum(axis=2),
    axis=1,
)
units_df["shank"] = contacts_df["shank"].to_numpy()[nearest_contact]
units_df

,x_mean,y_mean,shank
cluster,,,
18,12.412506,1007.089417,0
23,21.383923,1052.564941,0
26,13.031116,1065.637207,0
29,12.312404,1063.863647,0
32,11.823097,1076.413086,0
...,...,...,...
1063,261.985260,1021.237549,1
1064,265.271637,1021.180664,1
1077,269.083557,2080.519775,1


In [10]:
shank_gap = 4.0
cursor = 0.0
units_df["unit_display_y"] = np.nan

for shank, group in units_df.groupby("shank", sort=True):
    order = group.sort_values(["y_mean", "x_mean"], ascending=[True, True]).index
    units_df.loc[order, "unit_display_y"] = cursor + np.arange(len(order), dtype=float)
    cursor += len(order) + shank_gap

units_df = units_df.sort_values("unit_display_y")
units_df

,x_mean,y_mean,shank,unit_display_y
cluster,,,,
18,12.412506,1007.089417,0,0.0
23,21.383923,1052.564941,0,1.0
29,12.312404,1063.863647,0,2.0
26,13.031116,1065.637207,0,3.0
32,11.823097,1076.413086,0,4.0
...,...,...,...,...
1002,759.048462,1889.757935,3,313.0
1004,760.577087,1918.205200,3,314.0
1006,760.177917,1919.928345,3,315.0


In [11]:
units_df.to_dict()

{'x_mean': {18: 12.412506103515625,
  23: 21.383922576904297,
  29: 12.312403678894043,
  26: 13.031116485595703,
  32: 11.823097229003906,
  38: 22.392004013061523,
  47: 12.771273612976074,
  44: 12.572164535522461,
  46: 14.053526878356934,
  54: 15.24222183227539,
  56: 14.482929229736328,
  58: 19.55167007446289,
  57: 20.58995246887207,
  71: 21.329038619995117,
  77: 13.481398582458496,
  75: 18.85967254638672,
  79: 20.47562026977539,
  84: 13.879666328430176,
  83: 13.59065055847168,
  86: 21.091632843017578,
  94: 14.257217407226562,
  93: 13.676488876342773,
  100: 19.270584106445312,
  99: 17.333377838134766,
  107: 13.884851455688477,
  106: 17.326650619506836,
  112: 13.621302604675293,
  117: 14.029608726501465,
  115: 17.3695125579834,
  119: 17.37281036376953,
  124: 13.624433517456055,
  128: 17.3931941986084,
  127: 17.094858169555664,
  132: 14.046795845031738,
  133: 13.846923828125,
  134: 10.791486740112305,
  138: 13.687970161437988,
  140: 22.495018005371094,
 